In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

In [3]:
device = 'cpu'
batch_size = 64

Download the MNIST dataset and create separate DataLoaders for training and testing dataset.

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="/Users/shirindora/.datasets",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="/Users/shirindora/.datasets",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### **Task 1**: 

An autoencoder learns to compress an input into a compact **latent representation** and then reconstruct the original input from it — without using any labels. It has two parts:

- **Encoder**: fully connected layers that progressively reduce the dimensionality down to a small latent vector.
- **Decoder**: mirrors the encoder with fully connected layers that expand back to the original input size.

Each MNIST image is 28×28 = **784** pixels. Define an `Autoencoder` class with the following architecture:

**Encoder:** 784 → 256 → 64 → `latent_dim`

**Decoder:** `latent_dim` → 64 → 256 → 784

Use `ReLU` after every layer except the final decoder output. Use `latent_dim=32`. Expose separate `encode()` and `decode()` methods in addition to `forward()`.

Below is the code for the classifer we built in week 4 lab. Adapt it to built an autoencoder.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(28 * 28, 256)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fc3(x)
        return x

In [6]:
model = NeuralNetwork().to(device)

### **Task 2**: 

Below is the training loop we have used in one of the earlier weeks. Adapt it to train the autoencoder you built above.

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

In [8]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

### **Task 3**: Visualise reconstructions

Plot the original and reconstructed images side by side for 8 test examples. Then print the shape of the input, latent vector, and output to confirm the compression is working correctly.